In [0]:

# Platform Observability - Read-only discovery

catalog_schema = "workspace.video_game_sales"

tables_to_inspect = [
    "bronze_video_game_sales",
    "silver_video_game_sales",
    "data_quality_audit",
    "incremental_video_game_sales",
    "incremental_processing_audit",
    "gold_sales_by_year",
    "gold_sales_by_platform",
    "gold_sales_by_genre",
    "gold_sales_by_region",
    "gold_top_games"
]

for table_name in tables_to_inspect:
    full_table_name = f"{catalog_schema}.{table_name}"

    print(f"\n{'=' * 70}")
    print(f"TABLE: {full_table_name}")

    try:
        df = spark.table(full_table_name)

        print(f"Record count: {df.count()}")
        print("Schema:")
        df.printSchema()

    except Exception as error:
        print(f"Inspection failed: {str(error)[:500]}")

print("\nDISCOVERY COMPLETED")


In [0]:

from pyspark.sql import functions as F

from src.observability.monitoring import (
    collect_layer_metrics,
    build_reconciliation_metric,
    get_latest_dq_metrics
)

SCHEMA = "workspace.video_game_sales"

# 1. Current pipeline layer counts
layer_tables = {
    "BRONZE": "bronze_video_game_sales",
    "SILVER": "silver_video_game_sales",
    "INCREMENTAL_TARGET": "incremental_video_game_sales",
    "GOLD_YEAR": "gold_sales_by_year",
    "GOLD_PLATFORM": "gold_sales_by_platform",
    "GOLD_GENRE": "gold_sales_by_genre",
    "GOLD_REGION": "gold_sales_by_region",
    "GOLD_TOP_GAMES": "gold_top_games"
}

metrics = collect_layer_metrics(
    spark,
    SCHEMA,
    layer_tables
)

# 2. Bronze-to-Silver reconciliation
bronze_count = spark.table(
    f"{SCHEMA}.bronze_video_game_sales"
).count()

silver_count = spark.table(
    f"{SCHEMA}.silver_video_game_sales"
).count()

metrics.append(
    build_reconciliation_metric(
        "BRONZE_SILVER_RECONCILIATION",
        bronze_count,
        silver_count
    )
)

# 3. Silver-to-Incremental reconciliation
incremental_count = spark.table(
    f"{SCHEMA}.incremental_video_game_sales"
).count()

metrics.append(
    build_reconciliation_metric(
        "SILVER_INCREMENTAL_RECONCILIATION",
        silver_count,
        incremental_count
    )
)

# 4. Latest data quality audit status
dq_audit = spark.table(f"{SCHEMA}.data_quality_audit")

dq_metrics = get_latest_dq_metrics(dq_audit)

latest_run_id = dq_metrics["run_id"]
latest_dq_run = latest_run_id
failed_checks = dq_metrics["failed_checks"]
total_checks = dq_metrics["total_checks"]

if latest_run_id is not None:
    metrics.append((
        "LATEST_DQ_CHECKS",
        int(total_checks),
        "PASS" if failed_checks == 0 else "CHECK"
    ))

    metrics.append((
        "LATEST_DQ_FAILURES",
        int(failed_checks),
        "PASS" if failed_checks == 0 else "CHECK"
    ))
else:
    metrics.append((
        "LATEST_DQ_CHECKS",
        0,
        "CHECK"
    ))

# 5. Latest incremental processing audit
incremental_audit = spark.table(
    f"{SCHEMA}.incremental_processing_audit"
)

latest_incremental = (
    incremental_audit
    .orderBy(F.col("execution_timestamp").desc())
    .limit(1)
)

print("LATEST INCREMENTAL PROCESSING AUDIT")
display(latest_incremental)

# 6. Display monitoring snapshot
monitoring_df = spark.createDataFrame(
    metrics,
    ["metric_name", "metric_value", "status"]
)

print("PLATFORM MONITORING SNAPSHOT")
display(monitoring_df)

print("OBSERVABILITY SNAPSHOT COMPLETED")


In [0]:

from src.observability.monitoring import (
    summarize_dq_history,
    summarize_incremental_history
)

SCHEMA = "workspace.video_game_sales"

# 1. Historical data quality runs
dq_history = spark.table(
    f"{SCHEMA}.data_quality_audit"
)

dq_summary = summarize_dq_history(dq_history)

print("DATA QUALITY RUN HISTORY")
display(dq_summary)

# 2. Incremental processing history
incremental_history = spark.table(
    f"{SCHEMA}.incremental_processing_audit"
)

incremental_summary = summarize_incremental_history(
    incremental_history
)

print("INCREMENTAL PROCESSING HISTORY")
display(incremental_summary)

print("HISTORICAL MONITORING COMPLETED")


In [0]:

from pyspark.sql import functions as F

SCHEMA = "workspace.video_game_sales"

delta_tables = [
    "bronze_video_game_sales",
    "silver_video_game_sales",
    "incremental_video_game_sales",
    "gold_sales_by_year"
]

for table_name in delta_tables:
    full_name = f"{SCHEMA}.{table_name}"

    print(f"\nDELTA TRANSACTION HISTORY: {full_name}")

    try:
        history_df = spark.sql(
            f"DESCRIBE HISTORY {full_name} LIMIT 10"
        )

        display(
            history_df.select(
                "version",
                "timestamp",
                "operation",
                "operationParameters",
                "operationMetrics",
                "userMetadata"
            ).orderBy(F.col("version").desc())
        )

    except Exception as error:
        print(f"History unavailable: {str(error)[:500]}")

print("DELTA HISTORY INSPECTION COMPLETED")


In [0]:

from pyspark.sql import functions as F

SCHEMA = "workspace.video_game_sales"

monitored_tables = [
    "bronze_video_game_sales",
    "silver_video_game_sales",
    "incremental_video_game_sales",
    "gold_sales_by_year",
    "gold_sales_by_platform",
    "gold_sales_by_genre",
    "gold_sales_by_region",
    "gold_top_games"
]

history_frames = []

for table_name in monitored_tables:
    try:
        history = spark.sql(
            f"DESCRIBE HISTORY {SCHEMA}.{table_name}"
        )

        selected = history.select(
            F.lit(table_name).alias("table_name"),
            F.col("version"),
            F.col("timestamp"),
            F.col("operation"),
            F.col("operationMetrics")
        )

        history_frames.append(selected)

    except Exception as error:
        print(f"Could not inspect {table_name}: {str(error)[:300]}")

if history_frames:
    combined_history = history_frames[0]

    for frame in history_frames[1:]:
        combined_history = combined_history.unionByName(
            frame,
            allowMissingColumns=True
        )

    operations_monitor = (
        combined_history
        .withColumn(
            "execution_time_ms",
            F.col("operationMetrics").getItem("executionTimeMs").cast("long")
        )
        .withColumn(
            "rows_inserted",
            F.col("operationMetrics").getItem("numTargetRowsInserted").cast("long")
        )
        .withColumn(
            "rows_updated",
            F.col("operationMetrics").getItem("numTargetRowsUpdated").cast("long")
        )
        .withColumn(
            "rows_output",
            F.col("operationMetrics").getItem("numOutputRows").cast("long")
        )
        .withColumn(
            "files_added",
            F.col("operationMetrics").getItem("numTargetFilesAdded").cast("long")
        )
        .select(
            "table_name",
            "version",
            "timestamp",
            "operation",
            "execution_time_ms",
            "rows_inserted",
            "rows_updated",
            "rows_output",
            "files_added"
        )
        .orderBy(
            F.col("timestamp").desc(),
            F.col("table_name")
        )
    )

    print("DELTA OPERATIONS MONITOR")
    display(operations_monitor)

    print("OPERATION SUMMARY")
    display(
        operations_monitor
        .groupBy("table_name", "operation")
        .agg(
            F.count("*").alias("operation_count"),
            F.sum("rows_inserted").alias("total_inserted"),
            F.sum("rows_updated").alias("total_updated"),
            F.avg("execution_time_ms").alias("avg_execution_time_ms")
        )
        .orderBy("table_name")
    )

print("DELTA OPERATIONS MONITORING COMPLETED")


In [0]:
import plotly
import plotly.express as px

print("Plotly version:", plotly.__version__)
print("PLOTLY READY")

In [0]:

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Pipeline table health
health_data = [
    {"Layer": "Bronze", "Records": bronze_count},
    {"Layer": "Silver", "Records": silver_count},
    {"Layer": "Incremental", "Records": incremental_count},
    {
        "Layer": "Gold - Year",
        "Records": spark.table(f"{SCHEMA}.gold_sales_by_year").count()
    },
    {
        "Layer": "Gold - Platform",
        "Records": spark.table(f"{SCHEMA}.gold_sales_by_platform").count()
    },
    {
        "Layer": "Gold - Genre",
        "Records": spark.table(f"{SCHEMA}.gold_sales_by_genre").count()
    },
    {
        "Layer": "Gold - Region",
        "Records": spark.table(f"{SCHEMA}.gold_sales_by_region").count()
    },
    {
        "Layer": "Gold - Top Games",
        "Records": spark.table(f"{SCHEMA}.gold_top_games").count()
    }
]

health_fig = px.bar(
    health_data,
    x="Layer",
    y="Records",
    title="Data Platform - Table Record Counts",
    text="Records"
)

health_fig.update_layout(
    xaxis_title="Pipeline Layer",
    yaxis_title="Record Count",
    template="plotly_white"
)

health_fig.show()


# 2. Data quality run history
dq_chart_data = (
    dq_summary
    .select("run_id", "passed_checks", "failed_checks")
    .toPandas()
)

dq_chart_data["run_id"] = dq_chart_data["run_id"].str.slice(0, 8)

dq_fig = px.bar(
    dq_chart_data,
    x="run_id",
    y=["passed_checks", "failed_checks"],
    barmode="group",
    title="Data Quality - Pass/Fail by Run"
)

dq_fig.update_layout(
    xaxis_title="Data Quality Run",
    yaxis_title="Number of Checks",
    template="plotly_white"
)

dq_fig.show()


# 3. Incremental processing metrics
incremental_chart_data = (
    incremental_summary
    .select(
        "batch_id",
        "inserted_count",
        "updated_count",
        "unchanged_count"
    )
    .toPandas()
)

incremental_chart_data["batch_id"] = (
    incremental_chart_data["batch_id"].str.slice(0, 20)
)

incremental_fig = px.bar(
    incremental_chart_data,
    x="batch_id",
    y=["inserted_count", "updated_count", "unchanged_count"],
    barmode="group",
    title="Incremental Processing - Batch Metrics"
)

incremental_fig.update_layout(
    xaxis_title="Batch",
    yaxis_title="Record Count",
    template="plotly_white"
)

incremental_fig.show()


# 4. Delta MERGE execution time
merge_chart_data = (
    operations_monitor
    .filter(F.col("operation") == "MERGE")
    .select(
        "table_name",
        "version",
        "execution_time_ms",
        "rows_inserted",
        "rows_updated"
    )
    .toPandas()
)

merge_chart_data = merge_chart_data.dropna(
    subset=["execution_time_ms"]
)

merge_chart_data["merge_label"] = (
    merge_chart_data["table_name"]
    + " v"
    + merge_chart_data["version"].astype(str)
)

merge_fig = px.bar(
    merge_chart_data,
    x="merge_label",
    y="execution_time_ms",
    text="execution_time_ms",
    title="Delta Lake - MERGE Execution Time"
)

merge_fig.update_layout(
    xaxis_title="Delta Table Version",
    yaxis_title="Execution Time (ms)",
    template="plotly_white"
)

merge_fig.show()

print("PLATFORM OBSERVABILITY DASHBOARD GENERATED")


In [0]:

from pyspark.sql import functions as F
from datetime import datetime, timezone

SCHEMA = "workspace.video_game_sales"
MONITORING_TABLE = f"{SCHEMA}.platform_monitoring_history"

# Capture current table counts
tables = {
    "BRONZE": "bronze_video_game_sales",
    "SILVER": "silver_video_game_sales",
    "INCREMENTAL_TARGET": "incremental_video_game_sales",
    "GOLD_YEAR": "gold_sales_by_year",
    "GOLD_PLATFORM": "gold_sales_by_platform",
    "GOLD_GENRE": "gold_sales_by_genre",
    "GOLD_REGION": "gold_sales_by_region",
    "GOLD_TOP_GAMES": "gold_top_games"
}

snapshot_time = datetime.now(timezone.utc)

snapshot_rows = []

for metric_name, table_name in tables.items():
    record_count = spark.table(
        f"{SCHEMA}.{table_name}"
    ).count()

    snapshot_rows.append((
        metric_name,
        int(record_count),
        "AVAILABLE",
        snapshot_time
    ))

# Add reconciliation metrics
snapshot_rows.extend([
    (
        "BRONZE_SILVER_RECONCILIATION",
        int(silver_count - bronze_count),
        "PASS" if bronze_count == silver_count else "CHECK",
        snapshot_time
    ),
    (
        "SILVER_INCREMENTAL_RECONCILIATION",
        int(incremental_count - silver_count),
        "PASS" if incremental_count == silver_count else "CHECK",
        snapshot_time
    )
])

snapshot_schema = [
    "metric_name",
    "metric_value",
    "status",
    "snapshot_timestamp"
]

snapshot_df = spark.createDataFrame(
    snapshot_rows,
    snapshot_schema
)

# Append to a separate monitoring table
(
    snapshot_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(MONITORING_TABLE)
)

print("MONITORING SNAPSHOT SAVED")
print("Rows recorded:", snapshot_df.count())
print("Monitoring table:", MONITORING_TABLE)

display(
    spark.table(MONITORING_TABLE)
    .orderBy(F.col("snapshot_timestamp").desc())
)


In [0]:

from pyspark.sql import functions as F

SCHEMA = "workspace.video_game_sales"

alerts = []

# 1. Pipeline table availability
for table_name in [
    "bronze_video_game_sales",
    "silver_video_game_sales",
    "incremental_video_game_sales",
    "gold_sales_by_year",
    "gold_sales_by_platform",
    "gold_sales_by_genre",
    "gold_sales_by_region",
    "gold_top_games"
]:
    count = spark.table(f"{SCHEMA}.{table_name}").count()

    if count == 0:
        alerts.append(("EMPTY_TABLE", table_name, "CRITICAL"))

# 2. Layer reconciliation
if bronze_count != silver_count:
    alerts.append(("RECORD_MISMATCH", "BRONZE_TO_SILVER", "CRITICAL"))

if silver_count != incremental_count:
    alerts.append(("RECORD_MISMATCH", "SILVER_TO_INCREMENTAL", "CRITICAL"))

# 3. Latest data quality run
if latest_dq_run:
    if failed_checks > 0:
        alerts.append(("DATA_QUALITY_FAILURE", latest_run_id, "CRITICAL"))

# 4. Latest incremental processing status
latest_status = (
    incremental_audit
    .orderBy(F.col("execution_timestamp").desc())
    .first()["status"]
)

if str(latest_status).upper() != "SUCCESS":
    alerts.append(("INCREMENTAL_FAILURE", str(latest_status), "CRITICAL"))

# 5. Incremental reconciliation
latest_batch = incremental_summary.first()

if latest_batch["reconciliation_delta"] != 0:
    alerts.append(("INCREMENTAL_RECONCILIATION", latest_batch["batch_id"], "CRITICAL"))

# Display alert results
if alerts:
    alert_df = spark.createDataFrame(
        alerts,
        ["alert_type", "affected_component", "severity"]
    )
    print("PLATFORM ALERTS DETECTED")
    display(alert_df)
else:
    print("PLATFORM HEALTHY - NO ALERTS DETECTED")
    print("All configured monitoring rules passed.")


In [0]:

from datetime import datetime, timezone
from uuid import uuid4
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType
)

ALERT_TABLE = (
    "workspace.video_game_sales.platform_alert_history"
)

evaluation_id = str(uuid4())
evaluation_timestamp = datetime.now(timezone.utc)

alert_records = []

if alerts:
    for alert_type, component, severity in alerts:
        alert_records.append((
            evaluation_id,
            alert_type,
            component,
            severity,
            "OPEN",
            evaluation_timestamp
        ))
else:
    alert_records.append((
        evaluation_id,
        "PLATFORM_HEALTH_CHECK",
        "ALL_COMPONENTS",
        "INFO",
        "NO_ALERTS",
        evaluation_timestamp
    ))

alert_schema = StructType([
    StructField("evaluation_id", StringType(), False),
    StructField("alert_type", StringType(), False),
    StructField("affected_component", StringType(), False),
    StructField("severity", StringType(), False),
    StructField("alert_status", StringType(), False),
    StructField("evaluation_timestamp", TimestampType(), False)
])

alert_history_df = spark.createDataFrame(
    alert_records,
    schema=alert_schema
)

(
    alert_history_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(ALERT_TABLE)
)

print("ALERT EVALUATION PERSISTED")
print("Evaluation ID:", evaluation_id)
print("Records saved:", len(alert_records))
print("Alert history table:", ALERT_TABLE)

display(
    spark.table(ALERT_TABLE)
    .orderBy(F.col("evaluation_timestamp").desc())
)


In [0]:

import plotly.express as px
from pyspark.sql import functions as F

SCHEMA = "workspace.video_game_sales"

# Load persistent monitoring history
monitoring_history = spark.table(
    f"{SCHEMA}.platform_monitoring_history"
)

# 1. Pipeline record-count trends
volume_metrics = [
    "BRONZE",
    "SILVER",
    "INCREMENTAL_TARGET"
]

volume_pdf = (
    monitoring_history
    .filter(F.col("metric_name").isin(volume_metrics))
    .orderBy("snapshot_timestamp")
    .toPandas()
)

volume_fig = px.line(
    volume_pdf,
    x="snapshot_timestamp",
    y="metric_value",
    color="metric_name",
    markers=True,
    title="Pipeline Record Count Trends"
)

volume_fig.update_layout(
    xaxis_title="Monitoring Timestamp",
    yaxis_title="Record Count",
    template="plotly_white"
)

volume_fig.show()

# 2. Gold table volume
gold_pdf = (
    monitoring_history
    .filter(F.col("metric_name").startswith("GOLD_"))
    .orderBy("snapshot_timestamp")
    .toPandas()
)

gold_fig = px.bar(
    gold_pdf,
    x="metric_name",
    y="metric_value",
    color="metric_name",
    title="Gold Analytics Table Volumes"
)

gold_fig.update_layout(
    xaxis_title="Gold Table",
    yaxis_title="Record Count",
    template="plotly_white",
    showlegend=False
)

gold_fig.show()

# 3. Alert evaluation history
alert_history = spark.table(
    f"{SCHEMA}.platform_alert_history"
)

print("ALERT EVALUATION HISTORY")
display(
    alert_history.orderBy(
        F.col("evaluation_timestamp").desc()
    )
)

print("HISTORICAL DASHBOARD COMPLETED")
